# 01 - Manifest Review

Notebook read-only untuk meninjau manifest hasil proyek `pneumonia_reliability_study`.
Notebook ini TIDAK melakukan training model, TIDAK memodifikasi dataset, dan TIDAK
menghitung ulang split atau audit apapun - hanya membaca dan menampilkan hasil yang
sudah dibangun oleh script di `src/`.

Jalankan notebook ini dari folder `notebooks/` di dalam proyek (lokasi defaultnya).
Tidak ada path hardcoded - root proyek dicari otomatis dengan menelusuri folder induk
sampai ditemukan `src/project_config.py`.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd

def _find_project_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / "src" / "project_config.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root (looked for src/project_config.py "
        f"above {start}). Run this notebook from inside the project."
    )

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import project_config as cfg

MANIFESTS = cfg.get_manifests_dir()
REPORTS = cfg.get_reports_dir()
pd.set_option("display.max_columns", None)
print("Project root:", PROJECT_ROOT)

## Master manifest

In [ ]:
master = pd.read_csv(MANIFESTS / "master_manifest_derived.csv")
print(master.shape)
master.head()

## Tiga manifest utama (primary, cohort deduplicated 5.824 citra)

In [ ]:
original_dedup = pd.read_csv(MANIFESTS / "original_split_deduplicated.csv")
image_dedup = pd.read_csv(MANIFESTS / "image_stratified_split_deduplicated.csv")
grouped_dedup = pd.read_csv(MANIFESTS / "patient_grouped_split_deduplicated.csv")

for name, df in [("original_deduplicated", original_dedup),
                  ("image_stratified_deduplicated", image_dedup),
                  ("patient_grouped_deduplicated", grouped_dedup)]:
    print(name, df.shape, df["experimental_split"].value_counts().to_dict())

## Ringkasan split (6 strategi)

In [ ]:
split_summary = pd.read_csv(REPORTS / "split_summary.csv")
split_summary

## Laporan validasi sumber & split (primary vs sensitivity)

In [ ]:
with open(REPORTS / "source_validation_report.json") as f:
    source_report = json.load(f)
print("Source validation:", source_report["overall_status"])

with open(REPORTS / "split_validation_report.json") as f:
    split_report = json.load(f)
print("Primary analysis status:", split_report["primary_analysis_status"])
print("Sensitivity analysis status:", split_report["sensitivity_analysis_status"])

## Global deduplication keep-set & laporan deduplikasi

In [ ]:
keep_set = pd.read_csv(MANIFESTS / "global_deduplication_keep_set.csv")
print(keep_set["dedup_action"].value_counts())

dedup_report = pd.read_csv(REPORTS / "deduplication_report.csv")
dedup_report

## Catatan penting

- Test set pada manifest manapun **tidak boleh** dipakai untuk pemilihan model / hyperparameter tuning - hanya untuk evaluasi akhir satu kali.
- `patient_grouped_split_all_images.csv` (all-images) ditandai `primary_eligible = false` karena mengandung exact-duplicate MD5 yang tersebar lintas train/test (pasangan IM-0095/IM-0096). Gunakan `patient_grouped_split_deduplicated.csv` untuk analisis utama.